# 📅 WEEK 21 — DAY 7 — ENSEMBLE DEMO PROJECT

## 🏆 Project: Ensemble Model Comparison

### Models:
1. Decision Tree
2. Random Forest
3. AdaBoost
4. Gradient Boosting

### Dataset:

Breast Cancer Wisconsin Diagnostic Dataset

### Purpose:

Compare classification models using a consistent
train/test split and standard evaluation metrics.

## 1. IMPORT LIBRARIES

In [77]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

## 2. CREATE OUTPUT DIRECTORY

In [78]:
OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

## 3. LOAD DATASET

In [79]:
dataset = load_breast_cancer()

X = pd.DataFrame(
    dataset.data,
    columns = dataset.feature_names
)

y = pd.Series(dataset.target, name = "target")

print("="*60)
print("DATASET INFORMATION")
print("="*60)

print("Dataset shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nTarget mapping:")
for class_id, class_name in enumerate(dataset.target_names):
    print(f"{class_id}: {class_name}")

DATASET INFORMATION
Dataset shape: (569, 30)

Class distribution:
target
0    212
1    357
Name: count, dtype: int64

Target mapping:
0: malignant
1: benign


## 4. SPLIT DATA

In [80]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))


Training samples: 455
Test samples: 114


## 5. DEFINE MODELS

In [81]:
models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state = 42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators = 200,
        random_state = 42,
        n_jobs = -1
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators = 100,
        learning_rate = 0.5,
        random_state = 42,
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators = 100,
        learning_rate = 0.1,
        random_state = 42
    )
}

## 6. TRAIN AND EVALUATE MODELS

In [82]:
results = []
trained_models = {}

for model_name, model in models.items():

    print("\n" + "=" * 60)
    print(f"TRAINING: {model_name}")
    print("=" * 60)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        pos_label=0,
        zero_division=0,
    )

    recall = recall_score(
        y_test,
        predictions,
        pos_label=0,
        zero_division=0,
    )

    f1 = f1_score(
        y_test,
        predictions,
        pos_label=0,
        zero_division=0,
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    })

    trained_models[model_name] = model

    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Malignant", "Benign"],
        zero_division=0,
    ))

    print("Confusion matrix:")

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        labels=[0, 1],
        display_labels=["Malignant", "Benign"],
        cmap="Blues",
        colorbar=False,
    )

    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()

    safe_name = model_name.lower().replace(" ", "_")

    plt.savefig(
        OUTPUT_DIR / f"confusion_matrix_{safe_name}.png",
        dpi=150,
        bbox_inches="tight",
    )

    plt.close()


TRAINING: Decision Tree
              precision    recall  f1-score   support

   Malignant       0.85      0.93      0.89        42
      Benign       0.96      0.90      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114

Confusion matrix:

TRAINING: Random Forest
              precision    recall  f1-score   support

   Malignant       0.95      0.93      0.94        42
      Benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Confusion matrix:

TRAINING: AdaBoost
              precision    recall  f1-score   support

   Malignant       0.97      0.90      0.94        42
      Benign       0.95      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.9

## 7. CREATE COMPARISON TABLE

In [83]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by = "F1",
    ascending = False,
).reset_index(drop = True)

print("\n"+"="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

print(
    results_df.to_string(
        index = False,
        formatters = {
            "Accuracy": "{:.4f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1": "{:.4f}".format,
        },
    )
)

results_df.to_csv(
    OUTPUT_DIR / "model_comparison.csv",
    index = False,
)


FINAL MODEL COMPARISON
            Model Accuracy Precision Recall     F1
    Random Forest   0.9561    0.9512 0.9286 0.9398
         AdaBoost   0.9561    0.9744 0.9048 0.9383
Gradient Boosting   0.9561    0.9744 0.9048 0.9383
    Decision Tree   0.9123    0.8478 0.9286 0.8864


## 8. VISUALIZE MODEL PERFORMANCE

In [84]:
plot_df = results_df.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1"]
]

ax = plot_df.plot(
    kind = "bar",
    figsize = (12, 6)
)

ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_title("Model Comparision")
ax.set_ylim(0, 1.05)

plt.xticks(rotation = 20, ha = "right")
plt.legend(loc = "lower right")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "model_comparision.png",
    dpi = 150,
    bbox_inches = "tight"
)

plt.close()

## 9. FEATURE IMPORTANCE

In [85]:
forest_model = trained_models["Random Forest"]

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": forest_model.feature_importances_,
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False,
).head(15)

print("\nTop 15 Random Forest Features:")
print(importance_df.to_string(index=False))

plt.figure(figsize=(10, 7))

plt.barh(
    importance_df["Feature"][::-1],
    importance_df["Importance"][::-1],
)

plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "feature_importance.png",
    dpi=150,
    bbox_inches="tight",
)

plt.close()


Top 15 Random Forest Features:
             Feature  Importance
     worst perimeter    0.133100
          worst area    0.128052
worst concave points    0.108107
 mean concave points    0.094414
        worst radius    0.090639
         mean radius    0.058662
      mean perimeter    0.055242
           mean area    0.049938
      mean concavity    0.046207
     worst concavity    0.035357
          area error    0.034368
    mean compactness    0.018094
       worst texture    0.017869
   worst compactness    0.014481
        mean texture    0.014317


## 10. FINAL SUMMARY

In [86]:
print("\n" + "=" * 60)
print("EXPERIMENT COMPLETED")
print("=" * 60)

print("Results saved to:", OUTPUT_DIR.resolve())

print("\nGenerated files:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print("-", file_path.name)


EXPERIMENT COMPLETED
Results saved to: /home/mdzeeshan/agentic-ai-engineering-journey/phase_2_machine_learning/week_21_ensembles/day_7_ensemble_demo_project/notebook/outputs

Generated files:
- confusion_matrix_adaboost.png
- confusion_matrix_decision_tree.png
- confusion_matrix_gradient_boosting.png
- confusion_matrix_random_forest.png
- feature_importance.png
- model_comparision.png
- model_comparison.csv
